# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AndruMark/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

### Selected Lane: Lane 1 — Content Refresh Prioritization (Provisional)

Content decay is a persistent, silent failure mode for enterprise websites. Over time, search intent drifts, competitor coverage improves, and algorithmic updates shift SERP real estate away from dated articles. Content teams at scale face an intractable triage problem: an organization managing 10,000+ indexed URLs cannot manually inspect each page every quarter. Relying on naive age-based rules (e.g., "refresh anything older than 6 months") squanders editorial bandwidth on stable pages while missing high-traffic assets in active decline. By building an ML system that ranks pages by expected decline risk and business exposure, we transform an unmanageable catalog into an actionable weekly triage queue.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### Problem Framing
* **Research Question:** "Among published URLs with established search visibility, which pages are exhibiting early indicators of performance decay and represent the highest expected return on editorial intervention?"
* **Unit of Analysis:** A single URL (`page_path`) tracked across a rolling 90-day aggregation window within a client domain (`client_id`).
* **Model Output:** A calibrated probability $P(\text{is\_declining})$ paired with an exposure-weighted priority rank ($\text{Priority} = P(\text{declining}) \times \text{impressions\_90d}$) and primary reason codes (CTR degradation, position loss, staleness).
* **Operational Action:** Content teams extract the top-$K$ ranked URLs (e.g., $K=20$ pages/month) and assign staff writers targeted updates (updating data, aligning headings with SERP intent, adding missing subtopics).
* **Cost of a Wrong Call:**
  * *False Positive (Type I):* 4–6 hours of writer time wasted revising a stable page, yielding zero incremental traffic.
  * *False Negative (Type II):* Missing a high-visibility URL slipping from positions 1–3 down to striking distance, causing severe and compounding organic revenue loss.
* **Why ML Over Heuristics:** Simple rules ignore non-linear interactions; a 1-position drop from rank 2 to 3 destroys far more traffic than dropping from rank 20 to 25. A learned ranker captures multi-feature decay dynamics without arbitrary hardcoded thresholds.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import os
import pandas as pd
import numpy as np

# Load starter dataset
data_path = os.path.join("..", "..", "data", "raw", "content_refresh_anonymized.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "raw", "content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

if "is_declining_label" not in df.columns and "trend_direction" in df.columns:
    df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# 1. Total Universe & Baseline Decline Rate
total_pages = len(df)
total_clients = df["client_id"].nunique()
decline_rate = df["is_declining_label"].mean()

print(f"1. Opportunity Scale: {total_pages:,} pages across {total_clients} clients.")
print(f"   Baseline Decline Rate: {decline_rate:.1%} of pages are actively declining.")

# 2. CTR Cliff by Position Tier
tier_ctr = (
    df[df["impressions_90d"] >= 100]
    .groupby("position_tier")["ctr"]
    .mean()
    .sort_values(ascending=False)
)
print("\n2. The Non-Linear CTR Cliff across Position Tiers (Impressions >= 100):")
for tier, val in tier_ctr.items():
    print(f"   - {tier:12}: {val:.3f} average CTR")

# 3. Content Age vs Decline Correlation
median_age_declining = df[df["is_declining_label"] == 1]["content_age_days"].median()
median_age_healthy = df[df["is_declining_label"] == 0]["content_age_days"].median()
corr_age = df["content_age_days"].corr(df["is_declining_label"])

print(f"\n3. Inefficiency of Pure Age Rules:")
print(f"   - Median Content Age (Declining): {median_age_declining:.0f} days")
print(f"   - Median Content Age (Healthy):   {median_age_healthy:.0f} days")
print(f"   - Pearson correlation (age vs. decline): {corr_age:.3f}")
print("   --> Content age alone is a weak linear indicator; freshness heuristics miss recent decay.")


1. Opportunity Scale: 30,000 pages across 32 clients.
   Baseline Decline Rate: 54.2% of pages are actively declining.

2. The Non-Linear CTR Cliff across Position Tiers (Impressions >= 100):
   - page_1      : 0.355 average CTR
   - top_3       : 0.334 average CTR
   - striking    : 0.256 average CTR
   - page_3_5    : 0.142 average CTR
   - deep        : 0.055 average CTR

3. Inefficiency of Pure Age Rules:
   - Median Content Age (Declining): 216 days
   - Median Content Age (Healthy):   287 days
   - Pearson correlation (age vs. decline): -0.164
   --> Content age alone is a weak linear indicator; freshness heuristics miss recent decay.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### Claim Boundaries
* **What I CAN Claim:** 
  * Directional Prioritization: Multivariate signals (high baseline impressions, positional tier changes, CTR velocity) isolate decaying URLs significantly better than random inspection or static age heuristics.
  * Operational Utility: Learned models optimize Precision@$K$ on unseen client holdouts, directly saving editorial hours.
* **What I CANNOT Claim:** 
  * No Google Reverse-Engineering: We do not model or reverse-engineer Google’s ranking algorithms.
  * No Causal Traffic Guarantees: Re-optimizing content is an operational best practice, not a mathematical guarantee of ranking recovery against evolving competitors and search intent shifts.
  * No Universal Web Law: Insights reflect the audited multi-client dataset and do not generalize unconditionally to all search verticals.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.